# Card centering — YOLO OUTER / INNER box training

Run the cells top to bottom. Only **cell 1** normally needs editing.

1. Settings
2. Install, imports, mount Drive
3. Copy the dataset ZIP to local Colab disk and extract it
4. Helper functions
5. Label audit (catches swapped / broken labels)
6. Visualize random labeled images
7. Train (7b = resume after a Colab disconnect)
8. Training curves
9. Standard metrics (val + test)
10. **Centering accuracy on the test set** — the number that actually matters
11. Every test image: ground truth vs prediction, saved to Drive (worst first)
12. Export ONNX and check it matches PyTorch

In [ ]:
# =============================================================================
# 1. SETTINGS
# =============================================================================
from pathlib import Path
from datetime import datetime

DRIVE_ZIP = Path("/content/drive/MyDrive/card_centering/merged_centering_dataset_sep25.zip")
DRIVE_OUTPUT = Path("/content/drive/MyDrive/card_centering/runs_sep25")  # runs are saved here
LOCAL_DATA = Path("/content/data_centering")                             # fast local disk

# Optional: an older model to compare against on the test set (None to skip).
OLD_MODEL = Path("/content/drive/MyDrive/yolo8.onnx")
OLD_IMGSZ = 640

BASE_MODEL = "yolov8m.pt"  # "yolo11m.pt" is a drop-in alternative (same ONNX output format)
# Image size: bigger = more pixels on the thin card borders = more precise edges.
# MAXIMUM 896 for YOLOv8/YOLO11: these models can place a box edge at most 480 px
# (15 x stride 32) from the grid cell that detects it, so a box can be at most ~960 px.
# At 1024, 73% of the OUTER boxes in this dataset could NOT be predicted correctly.
IMGSZ = 832                # 640 = old model | 832 = 1.3x more pixels, with a safety margin
assert IMGSZ % 32 == 0 and IMGSZ <= 896, "IMGSZ must be a multiple of 32 and <= 896"

# Fixed-length training (no early stopping): the learning rate fully decays and the
# final CLOSE_MOSAIC epochs on whole cards always run. That is where the edges get refined.
EPOCHS = 100
CLOSE_MOSAIC = 15
BATCH = 16                 # A100: 16 | L4 / T4: 8
SEED = 42

CONF = 0.10                # low on purpose: we only keep the best OUTER and best INNER box
NAMES = {0: "OUTER", 1: "INNER"}

RUN_NAME = f"{Path(BASE_MODEL).stem}_{IMGSZ}_{datetime.now():%Y%m%d_%H%M}"

In [ ]:
# =============================================================================
# 2. INSTALL, IMPORTS, MOUNT DRIVE
# =============================================================================
!pip install -q "ultralytics==8.4.104" onnx onnxslim onnxruntime

import math
import random
import shutil
import zipfile

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import Image, display
from ultralytics import YOLO

from google.colab import drive
drive.mount("/content/drive")

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - training will be very slow")

In [ ]:
# =============================================================================
# 3. COPY DATASET ZIP TO LOCAL DISK AND EXTRACT
# =============================================================================
if not DRIVE_ZIP.is_file():
    raise FileNotFoundError(f"Dataset ZIP not found: {DRIVE_ZIP}")

shutil.rmtree(LOCAL_DATA, ignore_errors=True)
LOCAL_DATA.mkdir(parents=True)
local_zip = Path("/content") / DRIVE_ZIP.name

print("Copying ZIP from Drive...")
shutil.copy(DRIVE_ZIP, local_zip)
print("Extracting...")
with zipfile.ZipFile(local_zip) as archive:
    archive.extractall(LOCAL_DATA)
local_zip.unlink()

yaml_files = list(LOCAL_DATA.rglob("data.yaml"))
if len(yaml_files) != 1:
    raise RuntimeError(f"Expected exactly one data.yaml in the ZIP, found: {yaml_files}")
DATASET_ROOT = yaml_files[0].parent

names = yaml.safe_load(yaml_files[0].read_text())["names"]
names = dict(enumerate(names)) if isinstance(names, list) else names
if {int(k): str(v).upper() for k, v in names.items()} != NAMES:
    raise ValueError(f"Dataset classes {names} do not match {NAMES}")

# Absolute-path YAML so training works from any working directory.
DATA_YAML = DATASET_ROOT / "data_local.yaml"
DATA_YAML.write_text(yaml.safe_dump({
    "path": str(DATASET_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": NAMES,
}, sort_keys=False))

print("Dataset root:", DATASET_ROOT)
print(DATA_YAML.read_text())

In [ ]:
# =============================================================================
# 4. HELPERS
# =============================================================================
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
COLORS = {0: (255, 120, 0), 1: (0, 165, 255)}  # BGR: OUTER = blue, INNER = orange


def list_images(split):
    folder = DATASET_ROOT / "images" / split
    return sorted(p for p in folder.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)


def label_path(image_path):
    return DATASET_ROOT / "labels" / image_path.parent.name / f"{image_path.stem}.txt"


def read_labels(image_path, width=1, height=1):
    """Ground-truth boxes as {class_id: [array([x1, y1, x2, y2]), ...]} in pixels."""
    boxes = {}
    for line in label_path(image_path).read_text().splitlines():
        if line.strip():
            c, xc, yc, w, h = map(float, line.split())
            box = np.array([(xc - w / 2) * width, (yc - h / 2) * height,
                            (xc + w / 2) * width, (yc + h / 2) * height])
            boxes.setdefault(int(c), []).append(box)
    return boxes


def predict_boxes(model, image_bgr, imgsz=None, device=None):
    """Highest-confidence OUTER and INNER box as {class_id: array([x1, y1, x2, y2])} in pixels.
    Takes the already-loaded image, so prediction and ground truth use identical pixels."""
    result = model.predict(image_bgr, imgsz=imgsz or IMGSZ, conf=CONF,
                           device=DEVICE if device is None else device, verbose=False)[0]
    best = {}
    for box, conf, c in zip(result.boxes.xyxy.cpu().numpy(),
                            result.boxes.conf.cpu().numpy(),
                            result.boxes.cls.cpu().numpy().astype(int)):
        if c not in best or conf > best[c][1]:
            best[c] = (box, conf)
    return {c: box for c, (box, _) in best.items()}


def centering(outer, inner):
    """Border widths and centering. lr = left border share in % (50 = perfectly centered)."""
    left, top = inner[0] - outer[0], inner[1] - outer[1]
    right, bottom = outer[2] - inner[2], outer[3] - inner[3]
    return {
        "left": left, "right": right, "top": top, "bottom": bottom,
        "lr": 100 * left / (left + right) if left + right > 0 else float("nan"),
        "tb": 100 * top / (top + bottom) if top + bottom > 0 else float("nan"),
    }


def draw(image_bgr, boxes):
    """Draw {class_id: box} and return (RGB image, centering text)."""
    out = image_bgr.copy()
    thickness = max(2, round(max(out.shape[:2]) / 400))
    for c, box in boxes.items():
        x1, y1, x2, y2 = np.round(box).astype(int)
        cv2.rectangle(out, (x1, y1), (x2, y2), COLORS.get(c, (0, 0, 255)), thickness)
    text = "OUTER or INNER missing"
    if 0 in boxes and 1 in boxes:
        m = centering(boxes[0], boxes[1])
        text = f"L/R {m['lr']:.1f}/{100 - m['lr']:.1f}   T/B {m['tb']:.1f}/{100 - m['tb']:.1f}"
    return cv2.cvtColor(out, cv2.COLOR_BGR2RGB), text


def show_grid(panels, title, columns=4):
    """panels: list of (rgb_image, caption)."""
    rows = math.ceil(len(panels) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(5 * columns, 6.5 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, (rgb, caption) in zip(axes, panels):
        ax.imshow(rgb)
        ax.set_title(caption, fontsize=9)
    fig.suptitle(title, fontsize=14)
    fig.tight_layout()
    plt.show()
    plt.close(fig)

In [ ]:
# =============================================================================
# 5. LABEL AUDIT
# Every image must have exactly one OUTER and one INNER box, with INNER inside OUTER.
# =============================================================================
problems = []
for split in ["train", "val", "test"]:
    images = list_images(split)
    print(f"{split:5s}: {len(images)} images")
    for image_path in images:
        if not label_path(image_path).is_file():
            problems.append((split, image_path.name, "missing label file"))
            continue
        boxes = read_labels(image_path)
        counts = {c: len(boxes.get(c, [])) for c in NAMES}
        if set(boxes) - set(NAMES) or counts != {0: 1, 1: 1}:
            problems.append((split, image_path.name, f"box counts {counts}, classes {sorted(boxes)}"))
            continue
        m = centering(boxes[0][0], boxes[1][0])
        borders = [m["left"], m["right"], m["top"], m["bottom"]]
        if all(b < 0 for b in borders):
            problems.append((split, image_path.name, "OUTER and INNER look SWAPPED"))
        elif min(borders) < -0.001:
            problems.append((split, image_path.name, "INNER sticks out of OUTER"))

BAD_IMAGES = {name for _, name, _ in problems}
print(f"\nLabel problems: {len(problems)}")
for split, name, reason in problems:
    print(f"  [{split}] {name}: {reason}")
if problems:
    print("\nFix these in CVAT (or delete them) and re-zip: wrong labels teach the model wrong edges.")

In [ ]:
# =============================================================================
# 6. VISUALIZE RANDOM LABELED TRAINING IMAGES
# =============================================================================
random.seed(SEED)
panels = []
for image_path in random.sample(list_images("train"), 12):
    image = cv2.imread(str(image_path))
    h, w = image.shape[:2]
    boxes = {c: b[0] for c, b in read_labels(image_path, w, h).items()}
    rgb, text = draw(image, boxes)
    panels.append((rgb, f"{image_path.name[:45]}\n{text}"))
show_grid(panels, "Ground truth: blue = OUTER, orange = INNER")

In [ ]:
# =============================================================================
# 7. TRAIN
# Results go straight to Drive, so a Colab disconnect loses at most one epoch.
# =============================================================================
model = YOLO(BASE_MODEL)
model.train(
    data=str(DATA_YAML),
    imgsz=IMGSZ,
    epochs=EPOCHS,
    patience=0,        # 0 = no early stopping (see EPOCHS in cell 1)
    batch=BATCH,
    device=DEVICE,
    seed=SEED,
    project=str(DRIVE_OUTPUT),
    name=RUN_NAME,
    # Augmentation: keep boxes tight and axis-aligned.
    degrees=0.0,       # rotation makes the true box edges ambiguous
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,        # left/right flip is safe: labels flip with the image
    close_mosaic=CLOSE_MOSAIC,  # final epochs on whole cards (no mosaic) to refine the edges
    plots=True,
)

RUN_DIR = Path(model.trainer.save_dir)
BEST_PT = RUN_DIR / "weights" / "best.pt"
print("Best model:", BEST_PT)

In [ ]:
# =============================================================================
# 7b. RESUME AN INTERRUPTED TRAINING — run this INSTEAD of cell 7
# After a disconnect: run cells 1-4 first (they rebuild the dataset at the same
# local path), then this cell. It continues from the last saved epoch in the
# newest run folder on Drive and keeps writing into that same folder.
# =============================================================================
checkpoints = sorted(DRIVE_OUTPUT.glob("*/weights/last.pt"), key=lambda p: p.stat().st_mtime)
if not checkpoints:
    raise FileNotFoundError(f"No last.pt found under {DRIVE_OUTPUT}")
LAST_PT = checkpoints[-1]
print("Resuming from:", LAST_PT)

model = YOLO(str(LAST_PT))
model.train(resume=True, data=str(DATA_YAML), device=DEVICE, batch=BATCH)

RUN_DIR = Path(model.trainer.save_dir)
BEST_PT = RUN_DIR / "weights" / "best.pt"
print("Best model:", BEST_PT)

In [ ]:
# =============================================================================
# 8. TRAINING CURVES
# =============================================================================
for png in [RUN_DIR / "results.png", *sorted(RUN_DIR.glob("*PR_curve.png"))]:
    if png.is_file():
        print(png.name)
        display(Image(filename=str(png), width=1000))

In [ ]:
# =============================================================================
# 9. STANDARD METRICS ON VAL AND TEST
# mAP is almost saturated (~0.99) for this task, so also look at cell 10.
# =============================================================================
best_model = YOLO(str(BEST_PT))

for split in ["val", "test"]:
    metrics = best_model.val(data=str(DATA_YAML), split=split, imgsz=IMGSZ, batch=BATCH,
                             device=DEVICE, project=str(RUN_DIR), name=f"eval_{split}", exist_ok=True)
    print(f"\n{split.upper()}: mAP50-95 = {metrics.box.map:.4f} "
          f"(OUTER {metrics.box.maps[0]:.4f}, INNER {metrics.box.maps[1]:.4f})\n")

In [ ]:
# =============================================================================
# 10. CENTERING ACCURACY ON THE TEST SET
# lr_err / tb_err = error in centering percentage points
#   (e.g. truth 55/45, prediction 56/44  ->  error 1.0)
# edge_err_pct = worst box-edge error as % of card width
# =============================================================================
def centering_report(model, name, imgsz=None, device=None):
    rows = []
    for image_path in list_images("test"):
        if image_path.name in BAD_IMAGES:
            continue
        image = cv2.imread(str(image_path))
        h, w = image.shape[:2]
        truth = {c: b[0] for c, b in read_labels(image_path, w, h).items()}
        pred = predict_boxes(model, image, imgsz, device)
        row = {"image": image_path.name, "missing": not (0 in pred and 1 in pred)}
        if not row["missing"]:
            t, p = centering(truth[0], truth[1]), centering(pred[0], pred[1])
            edge_err = max(np.abs(pred[0] - truth[0]).max(), np.abs(pred[1] - truth[1]).max())
            row.update({
                "true_lr": t["lr"], "pred_lr": p["lr"], "lr_err": abs(p["lr"] - t["lr"]),
                "true_tb": t["tb"], "pred_tb": p["tb"], "tb_err": abs(p["tb"] - t["tb"]),
                "edge_err_pct": 100 * edge_err / (truth[0][2] - truth[0][0]),
            })
        rows.append(row)

    df = pd.DataFrame(rows)
    found = df[~df["missing"]]
    print(f"\n=== {name} ===")
    print(f"Test images: {len(df)} | OUTER or INNER missing: {int(df['missing'].sum())}")
    print(found[["lr_err", "tb_err", "edge_err_pct"]]
          .describe(percentiles=[0.5, 0.9, 0.95]).loc[["mean", "50%", "90%", "95%", "max"]].round(2))
    return df


results = centering_report(best_model, "NEW MODEL")
results.to_csv(RUN_DIR / "centering_test.csv", index=False)
print("\nSaved per-image results:", RUN_DIR / "centering_test.csv")

if OLD_MODEL is not None and Path(OLD_MODEL).is_file():
    old_model = YOLO(str(OLD_MODEL), task="detect")
    centering_report(old_model, f"OLD MODEL ({Path(OLD_MODEL).name})", imgsz=OLD_IMGSZ, device="cpu")

# Show the 6 worst test images: ground truth (left) vs prediction (right).
worst = results[~results["missing"]].assign(
    err=lambda d: d[["lr_err", "tb_err"]].max(axis=1)).nlargest(6, "err")
panels = []
for name in worst["image"]:
    image_path = DATASET_ROOT / "images" / "test" / name
    image = cv2.imread(str(image_path))
    h, w = image.shape[:2]
    truth_rgb, truth_text = draw(image, {c: b[0] for c, b in read_labels(image_path, w, h).items()})
    pred_rgb, pred_text = draw(image, predict_boxes(best_model, image))
    panels += [(truth_rgb, f"TRUTH {name[:30]}\n{truth_text}"), (pred_rgb, f"PREDICTED\n{pred_text}")]
show_grid(panels, "Worst centering errors: truth vs prediction")

In [ ]:
# =============================================================================
# 11. EVERY TEST IMAGE: GROUND TRUTH vs PREDICTION — saved to Drive
# Files are numbered worst-first: 001_... has the largest centering error.
# =============================================================================
SHOW_INLINE = True          # False = only save to Drive (faster, smaller notebook)
PANEL_HEIGHT = 900          # pixel height of each saved panel

COMPARE_DIR = RUN_DIR / "test_truth_vs_prediction"
shutil.rmtree(COMPARE_DIR, ignore_errors=True)
COMPARE_DIR.mkdir(parents=True)


def with_header(rgb, lines, height=PANEL_HEIGHT):
    """Resize to a fixed height and add white text lines on top."""
    scale = height / rgb.shape[0]
    image = cv2.resize(rgb, (round(rgb.shape[1] * scale), height))
    bar = np.full((32 * len(lines) + 12, image.shape[1], 3), 255, np.uint8)
    for i, line in enumerate(lines):
        cv2.putText(bar, line, (10, 30 + 32 * i), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 0, 0), 2, cv2.LINE_AA)
    return np.vstack([bar, image])


# Pass 1: predict every test image and measure the centering error.
records = []
for image_path in list_images("test"):
    image = cv2.imread(str(image_path))
    h, w = image.shape[:2]
    truth = {c: b[0] for c, b in read_labels(image_path, w, h).items()}
    pred = predict_boxes(best_model, image)
    error = float("inf")  # missing OUTER or INNER counts as worst
    if 0 in pred and 1 in pred and 0 in truth and 1 in truth:
        t, p = centering(truth[0], truth[1]), centering(pred[0], pred[1])
        error = max(abs(p["lr"] - t["lr"]), abs(p["tb"] - t["tb"]))
    records.append((error, image_path, truth, pred))

records.sort(key=lambda r: r[0], reverse=True)

# Pass 2: save side-by-side images to Drive, worst first.
for rank, (error, image_path, truth, pred) in enumerate(records, start=1):
    image = cv2.imread(str(image_path))
    truth_rgb, truth_text = draw(image, truth)
    pred_rgb, pred_text = draw(image, pred)
    error_text = "OUTER/INNER MISSING" if math.isinf(error) else f"centering error {error:.2f}"
    combined = np.hstack([
        with_header(truth_rgb, ["GROUND TRUTH", truth_text]),
        np.full((PANEL_HEIGHT + 76, 20, 3), 255, np.uint8),
        with_header(pred_rgb, [f"PREDICTION  ({error_text})", pred_text]),
    ])
    out_path = COMPARE_DIR / f"{rank:03d}_{image_path.stem}.jpg"
    cv2.imwrite(str(out_path), cv2.cvtColor(combined, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 90])

print(f"Saved {len(records)} comparisons to: {COMPARE_DIR}")

# Show them here too, at reduced size to keep the notebook light.
if SHOW_INLINE:
    for path in sorted(COMPARE_DIR.glob("*.jpg")):
        small = cv2.resize(cv2.imread(str(path)), None, fx=0.55, fy=0.55, interpolation=cv2.INTER_AREA)
        print(path.name)
        display(Image(data=cv2.imencode(".jpg", small)[1].tobytes()))

In [ ]:
# =============================================================================
# 12. EXPORT ONNX AND CHECK IT MATCHES PYTORCH
# Same settings as the previous export, except the image size is now IMGSZ.
# Your app must letterbox images to IMGSZ x IMGSZ before running this ONNX.
# =============================================================================
ONNX_PATH = Path(best_model.export(format="onnx", imgsz=IMGSZ, batch=1, dynamic=True,
                                   simplify=True, opset=12, nms=False))

onnx_model = YOLO(str(ONNX_PATH), task="detect")
differences = []
for image_path in random.sample(list_images("test"), 10):
    image = cv2.imread(str(image_path))
    pt_boxes = predict_boxes(best_model, image)
    onnx_boxes = predict_boxes(onnx_model, image, device="cpu")
    for c in NAMES:
        if c in pt_boxes and c in onnx_boxes:
            differences.append(np.abs(pt_boxes[c] - onnx_boxes[c]).max())
        elif (c in pt_boxes) != (c in onnx_boxes):
            print(f"{image_path.name}: {NAMES[c]} found by only one of PyTorch / ONNX")

if differences:
    print(f"\nLargest PyTorch vs ONNX box difference: {max(differences):.2f} px (expect < 1 px)")
else:
    print("\nNo boxes found to compare - check the model.")
print("ONNX model:", ONNX_PATH)
print("Run folder on Drive:", RUN_DIR)